# Step1: 기본구조및 파서기 이해하기

In [ ]:
!pip install python-dotenv langchain-openai langchain-core

In [ ]:
# json, list 파서기로 변형하여 봅니다.

from dotenv import load_dotenv                                   # ⓪ .env 로더
from langchain_openai import ChatOpenAI                          # ①
from langchain_core.prompts import ChatPromptTemplate            # ②
from langchain_core.output_parsers import StrOutputParser        # ③

# ⓪ .env 파일에서 API 키 불러오기
# .env에 OpenAI는  반드시 'OPENAI_API_KEY=키값'으로 정의되어 있어야함
load_dotenv()   # 같은 폴더의 .env 파일을 읽어 환경변수로 등록

# ① 모델 (Model)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
#   └ OPENAI_API_KEY는 load_dotenv()가 등록한 환경변수에서 자동 인식됨

# ② 프롬프트 (Prompt)
prompt = ChatPromptTemplate.from_template(
    "당신은 {role} 전문가입니다. {question}에 답해주세요."
)

# ③ 출력 파서 (Output Parser)
parser = StrOutputParser()

# ④ 체인 — 파이프로 연결 (LCEL)/프롬포트 받아서 에에엠에 넣어줘
chain = prompt | llm | parser

data={"role":"건강",
      "question":"여름철식중독"}
# ⑤ 실행
result = chain.invoke(data)
print(result)

여름철 식중독은 주로 고온다습한 날씨로 인해 발생하는 식품의 부패와 세균 증식으로 인해 발생합니다. 여름철에는 특히 다음과 같은 식중독 예방 수칙을 지키는 것이 중요합니다.

1. **식품 보관**: 냉장고의 온도를 4도 이하로 유지하고, 냉동고는 -18도 이하로 설정하세요. 음식은 가능한 한 빨리 냉장 보관하고, 상온에 방치하지 않도록 합니다.

2. **조리 전 손 씻기**: 음식을 조리하기 전과 후, 특히 생고기나 해산물을 다룬 후에는 반드시 손을 깨끗이 씻어야 합니다.

3. **조리 도구 소독**: 칼, 도마 등 조리 도구는 사용 후 깨끗이 세척하고 소독하여 교차 오염을 방지합니다.

4. **식품 익히기**: 고기, 생선, 계란 등은 충분히 익혀서 섭취해야 합니다. 내부 온도가 안전한 수준에 도달했는지 확인하세요.

5. **신선한 재료 사용**: 유통기한이 지난 식품이나 변질된 식품은 섭취하지 않도록 합니다. 신선한 재료를 사용하고, 조리 후에는 가능한 한 빨리 섭취합니다.

6. **음료수 관리**: 물이나 음료수는 깨끗한 물을 사용하고, 개봉 후에는 빠른 시간 내에 소비하는 것이 좋습니다.

7. **외식 시 주의**: 외식할 때는 위생 상태가 좋은 식당을 선택하고, 음식이 제대로 조리되었는지 확인합니다.

여름철에는 특히 식중독의 위험이 높으므로, 이러한 예방 수칙을 잘 지켜 건강한 여름을 보내시기 바랍니다. 만약 식중독 증상이 나타난다면 즉시 의료기관을 방문하는 것이 중요합니다.


##########################
# (참고) 관계형 db파일 만들기 및 text-to-SQL 랭체인
#############################


In [1]:
##########################
# [단계1] csv자료를 관계형 db로 만들기
#############################

import pandas as pd
import sqlite3

# 1. 준비된 CSV 파일 읽기
csv_file = "감기환자_진료데이터.csv"
df = pd.read_csv(csv_file)

# 2. 실제 파일 형태의 데이터베이스(.db) 연결 생성
# 실행하면 현재 폴더에 'medical_records.db' 파일이 실제로 만들어집니다.
db_file = "medical_records.db"
conn = sqlite3.connect(db_file)

# 3. 데이터프레임을 'patients_history'라는 테이블(시트)로 저장
# index=False는 판다스의 기본 일련번호(0, 1, 2...)를 DB에 저장하지 않는 설정입니다.
df.to_sql("./patients_history", conn, index=False, if_exists="replace")

# 4. 저장 완료 후 안전하게 연결 닫기
conn.close()

print(f"🎉 성공: '{csv_file}' 데이터가 '{db_file}' 파일로 영구 저장되었습니다!")

🎉 성공: '감기환자_진료데이터.csv' 데이터가 'medical_records.db' 파일로 영구 저장되었습니다!


In [2]:
import os
import sqlite3
import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 1. 프롬프트 6요소(맥락, 작업, 지시, 입력구조, 범위, 출력형식)를 반영한 고도화 프롬프트
system_prompt = """
[1. Context (맥락)]
너는 병원 진료 기록이 저장된 SQLite 데이터베이스를 관리하는 최고의 데이터 분석가이자 AI 엔지니어다.

[2. Task (작업)]
사용자의 자연어 질문을 분석하여, 데이터베이스에서 정확한 통계나 정보를 조회할 수 있는 최적의 SQLite 쿼리문(SELECT)을 생성하라.

[3. Input & Schema (입력 구조)]
조회할 테이블 이름은 'patients_history'이며, 컬럼 명세와 데이터의 실제 의미는 다음과 같다:
- 환자ID, 진료일자, 이름, 나이, 성별, 체온, 증상지속일, 처방, 진료구분, 진료내용
- 증상 (예: '두통/인후통/재채기', '오한/피로감/기침') -> 환자가 병원에 와서 호소한 주관적 '느낌/현상' 글
- 진단명 (예: '급성 비인두염', '급성 기관지염', '독감 의심', '상기도 감염') -> 의사가 최종적으로 내린 '병명/진단' 결과

[4. Instructions (상세 지시사항)]
- 사용자가 '독감 환자', '독감 의심 환자', '상기도 감염 환자' 등 구체적인 '병명'이나 '진단 결과'를 물어볼 때는, 반드시 '증상' 컬럼이 아닌 [진단명] 컬럼을 기준으로 조건절(WHERE)을 작성해야 한다.
- 예: '독감 의심 환자' -> WHERE 진단명 = '독감 의심' 또는 WHERE 진단명 LIKE '%독감%'

[5. Scale & Constraints (범위 및 제약조건)]
- 생각, 해설, 부연설명은 절대 출력하지 마라.
- 마크다운 문법 기호(```sql ... ```)는 절대 포함하지 마라.
- SQL 구문 내부에서 문자열 상수를 비교할 때는 반드시 싱글 쿼테이션(예: '독감 의심')을 사용하라.

[6. Output Format (출력 형식)]
오직 'SELECT'로 시작하여 세미콜론(';')으로 끝나는 실행 가능한 순수 SQL 한 줄 문자열만 반환하라.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}")
])

# 2. 파이프라인 조립 (LCEL)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
output_parser = StrOutputParser()
text_to_sql_chain = prompt | llm | output_parser

# 3. 자연어 질문 던지기 및 SQL 생성
user_question = "60세 이상 환자 중에서 독감 의심 환자는 몇 명이야?"
print(f"🗣️ 유저 질문: {user_question}\n")

generated_sql = text_to_sql_chain.invoke({"question": user_question})
print(f"🤖 [AI 생성 쿼리]: {generated_sql}\n")
print("-" * 60)


# 4. 실제 DB 파일('medical_records.db') 검증 및 결과 꺼내오기
db_file = "medical_records.db"
csv_file = "감기환자_진료데이터.csv"

is_ready = True
conn = sqlite3.connect(db_file)
cursor = conn.cursor()
cursor.execute(
    "SELECT count(name) FROM sqlite_master WHERE type='table' AND name='patients_history'"
)

if cursor.fetchone()[0] == 0:
    print(f"⚠️ [안내] '{db_file}' 내에 테이블이 없어 '{csv_file}' 파일로 자동 빌드를 시도합니다...")
    if os.path.exists(csv_file):
        df_csv = pd.read_csv(csv_file)
        df_csv.to_sql("patients_history", conn, index=False, if_exists="replace")
        print("-> [성공] 테이블 생성 및 데이터 복구가 완료되었습니다.\n")
    else:
        print(f"❌ 오류: 복구할 원본 '{csv_file}' 파일이 현재 폴더에 존재하지 않습니다.")
        print("💡 해결책: CSV 파일을 현재 주피터 노트북 파일과 같은 폴더에 넣어주세요.")
        is_ready = False

# 5. 안전한 부근에서만 조회 실행
if is_ready:
    print(f"⚙️ [시스템 내부 동작] 데이터베이스에 실행되는 쿼리:\n    {generated_sql}\n")
    result_df = pd.read_sql(generated_sql, conn)
    print("📊 [최종 시스템 조회 결과]")
    print(result_df)
else:
    print("\n🛑 시스템을 가동할 수 없어 조회를 중단합니다. 파일을 확인해 주세요.")

conn.close()

🗣️ 유저 질문: 60세 이상 환자 중에서 독감 의심 환자는 몇 명이야?

🤖 [AI 생성 쿼리]: SELECT COUNT(*) FROM patients_history WHERE 진단명 = '독감 의심' AND 나이 >= 60;

------------------------------------------------------------
⚠️ [안내] 'medical_records.db' 내에 테이블이 없어 '감기환자_진료데이터.csv' 파일로 자동 빌드를 시도합니다...
-> [성공] 테이블 생성 및 데이터 복구가 완료되었습니다.

⚙️ [시스템 내부 동작] 데이터베이스에 실행되는 쿼리:
    SELECT COUNT(*) FROM patients_history WHERE 진단명 = '독감 의심' AND 나이 >= 60;

📊 [최종 시스템 조회 결과]
   COUNT(*)
0         1


In [4]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
from langchain_core.output_parsers import JsonOutputParser

####################################
# 조건 분기 체인 — 판정을 JSON으로 받기
####################################

json_parser = JsonOutputParser()   # JSON 파서

# 1단계: 위험도 판정 (JSON으로 답하게 지시)
judge = ChatPromptTemplate.from_template(
    "다음 증상을 분석해서 반드시 JSON으로만 답해. 다른 설명은 절대 넣지 마.\n"
    '형식: {{"level": "응급 또는 경미", "reason": "판단 이유 한 문장"}}\n'
    "증상:\n{text}"
)

emergency = ChatPromptTemplate.from_template(
    "다음 증상은 응급 상황이야. 119에 연락하고 병원에 가라는 강한 경고문을 써줘:\n{text}"
)
mild = ChatPromptTemplate.from_template(
    "다음 증상은 경미해. 집에서 할 수 있는 관리법을 친절하게 안내해줘:\n{text}"
)

# ── 입력 ──
symptom = "갑자기 가슴이 심하게 조이고 왼팔이 저리며 식은땀이 난다"

# 1단계: JSON으로 판정 받기
verdict = (judge | llm | json_parser).invoke({"text": symptom})
print("【판정 결과(JSON)】")
print(verdict, "\n")
# 예: {'level': '응급', 'reason': '심근경색 의심 증상입니다'}

# 2단계: JSON의 level 값으로 분기
if verdict["level"] == "응급":
    result = (emergency | llm | parser).invoke({"text": symptom})
else:
    result = (mild | llm | parser).invoke({"text": symptom})

print("【안내】")
print(result)

【판정 결과(JSON)】
{'level': '응급', 'reason': '가슴 통증과 왼팔 저림, 식은땀은 심장 문제의 징후일 수 있어 즉각적인 의료 조치가 필요하다.'} 

【안내】
⚠️ 긴급 경고 ⚠️

갑자기 가슴이 심하게 조이고, 왼팔이 저리며 식은땀이 나는 증상이 나타났다면, 이는 매우 심각한 응급 상황일 수 있습니다! 즉시 119에 연락하여 구급차를 요청하고, 가능한 한 빨리 병원으로 가야 합니다.

이러한 증상은 심장 문제나 다른 생명을 위협하는 상태의 신호일 수 있으므로, 절대 지체하지 말고 즉각적인 조치를 취하세요. 당신의 안전이 최우선입니다!


In [7]:
########################
# 정규식 - 기밀정보 마스킹 처리
############################
import re
import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template(
    "다음 진료기록을 환자가 이해하기 쉽게 설명해줘:\n{text}"
)
chain = prompt | llm | parser

# ── 마스킹 함수 ──
def mask_privacy(text):
    text = re.sub(r"\d{6}-\d{7}", "******-*******", text)        # 주민번호
    text = re.sub(r"01\d-\d{3,4}-\d{4}", "010-****-****", text)  # 전화번호
    text = re.sub(r"환자:\s*\S+", "환자: ***", text)             # 이름
    return text

# ── Gradio 처리 함수 ──
def process(record):
    masked = mask_privacy(record)          # 1. 마스킹
    answer = chain.invoke({"text": masked}) # 2. 마스킹된 걸로 모델 호출
    return masked, answer                   # 3. 둘 다 반환

demo = gr.Interface(
    fn=process,
    inputs=gr.Textbox(lines=8, label="진료기록 원본 (주민번호·전화·이름 포함)"),
    outputs=[
        gr.Textbox(lines=8, label="🔒 마스킹 결과 (모델에 실제 전달된 내용)"),
        gr.Textbox(lines=8, label="📋 환자 안내문 (최종 결과)"),
    ],
    title="진료기록 안전 변환기",
    description="민감정보를 가린 뒤 모델에 전달합니다. 왼쪽에서 마스킹 결과를 직접 확인하세요.",
)
demo.launch()

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [6]:
%pip install gradio


   ---------------------------------------- 0.0/31.4 MB ? eta -:--:--
   ------- -------------------------------- 6.0/31.4 MB 33.5 MB/s eta 0:00:01
   ------------------ --------------------- 14.7/31.4 MB 38.4 MB/s eta 0:00:01
   --------------------------------- ------ 26.2/31.4 MB 43.7 MB/s eta 0:00:01
   ---------------------------------------  31.2/31.4 MB 43.0 MB/s eta 0:00:01
   ---------------------------------------- 31.4/31.4 MB 35.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


#### 환자: 김민수 (50세 남성)
#### 주민등록번호: 750312-1234567
#### 연락처: 010-1234-5678
#### 진단: 급성 위염

# Step2. 체인이어가기 (순차체인)

In [8]:
# ── 준비 ──
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# ① 모델
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ② 파서
parser = StrOutputParser()

# ③ 단계별 프롬프트
prompt1 = ChatPromptTemplate.from_template(
    "당신은 {role} 전문가입니다. {question}에 답해주세요."
)
prompt2 = ChatPromptTemplate.from_template(
    "다음 내용을 딱 3줄로 요약해줘:\n{text}"
)
prompt3 = ChatPromptTemplate.from_template(
    "다음 한국어를 영어로 번역해줘:\n{text}"
)

# ④ 단계별 체인
chain1 = prompt1 | llm | parser
chain2 = prompt2 | llm | parser
chain3 = prompt3 | llm | parser

# ⑤ 단계별 실행 + 출력
data = {"role": "건강", "question": "여름철식중독"}

step1 = chain1.invoke(data)
print("=" * 40)
print("【1단계】 전문가 답변")
print("=" * 40)
print(step1)

step2 = chain2.invoke({"text": step1})
print("\n" + "=" * 40)
print("【2단계】 3줄 요약")
print("=" * 40)
print(step2)

step3 = chain3.invoke({"text": step2})
print("\n" + "=" * 40)
print("【3단계】 영어 번역")
print("=" * 40)
print(step3)

【1단계】 전문가 답변
여름철 식중독은 주로 고온다습한 날씨로 인해 발생하는 식품의 부패와 세균 증식으로 인해 발생합니다. 여름철에는 특히 다음과 같은 식중독 예방 수칙을 지키는 것이 중요합니다.

1. **식품 보관**: 음식은 적절한 온도에서 보관해야 합니다. 냉장고는 4도 이하, 냉동고는 -18도 이하로 유지하세요. 특히 육류, 해산물, 유제품은 신선도를 유지하기 위해 빠르게 냉장 보관해야 합니다.

2. **조리 전 손 씻기**: 음식을 조리하기 전과 후, 특히 생고기나 해산물을 만진 후에는 반드시 손을 깨끗이 씻어야 합니다.

3. **조리 온도**: 고기와 해산물은 충분히 익혀야 합니다. 일반적으로 고기는 내부 온도가 75도 이상이 되어야 안전합니다.

4. **음식 재사용 주의**: 남은 음식을 재사용할 경우, 빠른 시간 내에 냉장 보관하고, 다시 데울 때는 충분히 가열해야 합니다.

5. **신선한 재료 사용**: 여름철에는 신선한 과일과 채소를 섭취하는 것이 좋지만, 세척이 불충분하면 오히려 식중독의 원인이 될 수 있습니다. 흐르는 물에 깨끗이 씻어주세요.

6. **음료수 관리**: 얼음이나 음료수도 청결한 물로 만들어야 하며, 외부에서 구입한 음료수는 신뢰할 수 있는 곳에서 구매하는 것이 좋습니다.

7. **외식 시 주의**: 외식할 때는 위생 상태가 좋은 식당을 선택하고, 음식이 제대로 조리되었는지 확인하세요.

여름철 식중독은 예방이 가장 중요합니다. 위의 수칙을 잘 지켜 건강한 여름을 보내시기 바랍니다!

【2단계】 3줄 요약
여름철 식중독은 고온다습한 날씨로 인해 발생하며, 예방을 위해 몇 가지 수칙을 지켜야 합니다. 음식은 적절한 온도에서 보관하고, 조리 전후에 손을 씻으며, 고기와 해산물은 충분히 익혀야 합니다. 또한 신선한 재료를 사용하고, 외식 시 위생 상태를 확인하는 것이 중요합니다.

【3단계】 영어 번역
Summer food poisoning occurs due to hot and humid weather, and se

In [9]:
################################
# 위의코드를 for 문으로 줄이기
####################################

# ── 프롬프트 "문자열"만 리스트에 담기 ──
templates = [
    "당신은 {role} 전문가입니다. {question}에 답해주세요.",
    "다음 내용을 딱 3줄로 요약해줘:\n{text}",
    "다음 한국어를 영어로 번역해줘:\n{text}",
]

result = {"role": "건강", "question": "여름철식중독"}

# ── 반복문 안에서 from_template 한 번만 호출 ──
for i, template in enumerate(templates, start=1):
    prompt = ChatPromptTemplate.from_template(template)   # ← 여기서 한 번
    chain = prompt | llm | parser

    if i == 1:
        result = chain.invoke(result)
    else:
        result = chain.invoke({"text": result})

# print를 for안에 넣으면 1,2,3단계모두 출력
print(f"【{i}단계】")
print(result, "\n")

【3단계】
Summer food poisoning occurs due to hot and humid weather. To prevent it, it is important to practice hygienic handwashing, proper food storage, sufficient cooking, use fresh ingredients, manage beverages, and check the smell and color of food. In particular, cooked food should be refrigerated within 2 hours, and meat and seafood should be cooked to an internal temperature of at least 75 degrees Celsius. If symptoms of food poisoning appear, one should visit a medical institution immediately. 



In [10]:
#####################################################
# 순차체인:
# 1단계: 진료기록 → 환자용 쉬운 설명
# 2단계: 핵심 수칙 3개만 추출
# 3단계: 수칙 → 외우기 쉬운 노래 가사
###############################################
# ── 준비 ──
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)  # 가사엔 약간의 창의성
parser = StrOutputParser()

# ── 가상의 진료기록 (※ 실제 환자정보 아님, 교육용 샘플) ──
record = """
환자: 50세 남성
진단: 급성 위염 (Acute gastritis)
처방: PPI 제제 1일 1회 아침 식전 복용, 제산제 식후 30분
소견: 상복부 통증 및 속쓰림 호소. 자극적 음식, 음주, 카페인 제한 권고.
2주 후 증상 지속 시 내시경 검사 예정.
"""

# ── 단계 정의: (켜짐, 템플릿) ──
steps = [
    # 1단계: 진료기록 → 환자용 쉬운 설명
    (True,
     "다음 진료기록을 의학 지식이 없는 환자도 이해할 수 있게 "
     "쉽고 친절하게 풀어서 설명해줘:\n{text}"),

    # 2단계: 핵심 수칙 3개만 추출
    (True,
     "다음 내용에서 환자가 꼭 지켜야 할 핵심 수칙 3가지만 "
     "짧게 뽑아줘:\n{text}"),

    # 3단계: 수칙 → 외우기 쉬운 노래 가사
    (True,
     "다음 수칙들을 외우기 쉽고 리듬감 있는 짧은 노래 가사로 "
     "만들어줘. 따라 부르기 좋게 운율을 살려줘:\n{text}"),
]

# ── 실행 ──
result = {"text": record}
first = True

for on, template in steps:
    if not on:
        continue
    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | llm | parser
    if first:
        result = chain.invoke(result)
        first = False
    else:
        result = chain.invoke({"text": result})

# ── 최종 결과: 노래 가사 ──
print(result)

**(Verse 1)**  
아침에 일어나서, PPI 한 번,  
식사 전에 챙겨, 건강을 지켜보자.  

**(Chorus)**  
하루에 한 번, 아침 꼭 챙기고,  
PPI 제제, 잊지 말고,  
우리의 건강, 함께 지켜가요,  
리듬에 맞춰, 따라 부르세요!  

**(Verse 2)**  
식사 후 30분, 제산제를 챙기고,  
소화가 더 편해, 기분이 좋아지죠.  

**(Chorus)**  
하루에 한 번, 아침 꼭 챙기고,  
PPI 제제, 잊지 말고,  
우리의 건강, 함께 지켜가요,  
리듬에 맞춰, 따라 부르세요!  

**(Bridge)**  
자극적인 음식, 음주 줄이기,  
카페인도 조심해, 건강을 위해!  

**(Chorus)**  
하루에 한 번, 아침 꼭 챙기고,  
PPI 제제, 잊지 말고,  
우리의 건강, 함께 지켜가요,  
리듬에 맞춰, 따라 부르세요!  

**(Outro)**  
건강한 생활, 모두 함께해요,  
이 수칙들로, 행복해져요!


# Step3. RunnableLambda를 쓴 체인

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template("{topic}에 대해 길게 설명해줘.")

# RunnableLambda 없이 일반 람다 함수만 넣어도 잘 동작합니다.
chain = (
    (lambda x: {"topic": x.strip()})   # <- 자동 변환됨
    | prompt
    | llm
    | parser
)

# 스트리밍도 문제없이 작동합니다.
for chunk in chain.stream("   HTN예방   "):
    print(chunk, end="", flush=True)

HTN(고혈압, Hypertension)은 심혈관 질환의 주요 위험 요소 중 하나로, 심장병, 뇌졸중, 신장 질환 등 다양한 건강 문제를 유발할 수 있습니다. 따라서 HTN 예방은 건강한 삶을 유지하는 데 매우 중요합니다. 아래에서는 HTN 예방을 위한 여러 가지 방법에 대해 자세히 설명하겠습니다.

### 1. 건강한 식습관
- **저염식**: 나트륨 섭취를 줄이는 것이 중요합니다. 가공식품, 패스트푸드, 소금이 많이 들어간 음식은 피하고, 신선한 과일과 채소를 많이 섭취하는 것이 좋습니다.
- **균형 잡힌 식사**: 과일, 채소, 통곡물, 저지방 유제품, 살코기 등을 포함한 균형 잡힌 식사를 통해 영양소를 고르게 섭취해야 합니다.
- **칼륨 섭취**: 칼륨은 혈압을 낮추는 데 도움을 줄 수 있습니다. 바나나, 감자, 시금치, 아보카도 등 칼륨이 풍부한 음식을 섭취하는 것이 좋습니다.

### 2. 규칙적인 운동
- **신체 활동**: 주 150분 이상의 중등도 유산소 운동(걷기, 자전거 타기 등)을 권장합니다. 운동은 체중 조절, 스트레스 감소, 심혈관 건강 개선에 도움을 줍니다.
- **근력 운동**: 주 2회 이상 근력 운동을 포함하여 전반적인 체력을 강화하는 것도 중요합니다.

### 3. 체중 관리
- **건강한 체중 유지**: 비만은 고혈압의 주요 원인 중 하나입니다. 적정 체중을 유지하기 위해 식사 조절과 운동을 병행해야 합니다.
- **체중 감량**: 과체중인 경우, 체중을 줄이는 것만으로도 혈압을 낮출 수 있습니다.

### 4. 스트레스 관리
- **스트레스 감소**: 스트레스는 혈압을 상승시킬 수 있으므로, 명상, 요가, 심호흡 등의 방법으로 스트레스를 관리하는 것이 중요합니다.
- **취미 활동**: 자신이 좋아하는 활동을 통해 스트레스를 해소하고 정신적인 안정을 찾는 것도 도움이 됩니다.

### 5. 음주 및 흡연 관리
- **금연**: 흡연은 혈압을 높이고 심혈관 질환의 위험을 증가시킵니다. 금연은 건강을 개선하는 데 큰 도움이

In [12]:
####################
# RunnableLambda
########################
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
# 1. 런에이블람다 임포트
from langchain_core.runnables import RunnableLambda

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template("{topic}에 대해 길게 설명해줘.")

# 2. 명시적으로 RunnableLambda를 적용한 체인 연결
chain = (
    RunnableLambda(lambda x: {"topic": x.strip()})   # <- 런에이블로 감싸준 함수
    | prompt
    | llm
    | parser
)

for chunk in chain.stream("   HTN예방   "):
    print(chunk, end="", flush=True)

HTN(고혈압, Hypertension)은 심혈관 질환의 주요 위험 요소 중 하나로, 심장병, 뇌졸중, 신장 질환 등 다양한 건강 문제를 유발할 수 있습니다. 따라서 HTN 예방은 건강한 삶을 유지하는 데 매우 중요합니다. 다음은 HTN 예방을 위한 여러 가지 방법에 대한 자세한 설명입니다.

### 1. 건강한 식습관
- **저염식**: 나트륨 섭취를 줄이는 것이 중요합니다. 가공식품, 패스트푸드, 소금이 많이 들어간 음식은 피하고, 신선한 과일과 채소를 많이 섭취하는 것이 좋습니다.
- **균형 잡힌 식사**: 과일, 채소, 통곡물, 저지방 유제품, 살코기 등을 포함한 균형 잡힌 식사를 통해 영양소를 고르게 섭취해야 합니다.
- **칼륨 섭취 증가**: 칼륨은 혈압을 낮추는 데 도움을 줄 수 있습니다. 바나나, 감자, 시금치, 아보카도 등 칼륨이 풍부한 음식을 섭취하는 것이 좋습니다.

### 2. 규칙적인 운동
- **신체 활동**: 주 150분 이상의 중등도 유산소 운동(걷기, 자전거 타기 등)을 권장합니다. 운동은 체중 조절, 스트레스 감소, 심혈관 건강 개선에 도움을 줍니다.
- **근력 운동**: 주 2회 이상 근력 운동을 포함하여 전반적인 체력을 강화하는 것도 중요합니다.

### 3. 체중 관리
- **건강한 체중 유지**: 비만은 고혈압의 주요 원인 중 하나입니다. 적정 체중을 유지하기 위해 식사 조절과 운동을 병행해야 합니다.
- **체중 감량**: 과체중인 경우, 체중을 줄이는 것만으로도 혈압을 낮출 수 있습니다.

### 4. 스트레스 관리
- **스트레스 감소**: 만성 스트레스는 혈압을 높일 수 있습니다. 명상, 요가, 심호흡, 취미 활동 등을 통해 스트레스를 관리하는 방법을 찾아야 합니다.
- **사회적 지원**: 친구나 가족과의 긍정적인 관계를 유지하는 것도 스트레스 감소에 도움이 됩니다.

### 5. 음주 및 흡연 관리
- **금연**: 흡연은 혈압을 높이고 심혈관 질환의 위험을 증가시킵니다. 금연은 건강을 개선하는 가장 중요

In [13]:
import re
import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# 1. 컴포넌트 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template(
    "다음 진료기록을 환자가 이해하기 쉽게 설명해줘:\n{text}"
)

# 2. 순수 파이썬 함수 정의
def mask_privacy(text):
    """민감정보를 가리는 마스킹 함수"""
    text = re.sub(r"\d{6}-\d{7}", "******-*******", text)        # 주민번호
    text = re.sub(r"01\d-\d{3,4}-\d{4}", "010-****-****", text)  # 전화번호
    text = re.sub(r"환자:\s*\S+", "환자: ***", text)             # 이름
    return text

# 3. RunnableLambda를 사용한 체인 패키징
# - 시작점(mask_privacy)은 문자열을 받아 문자열을 리턴하므로 RunnableLambda로 감싸줍니다.
# - 중간에 람다를 넣어 프롬프트가 원하는 딕셔너리 형태 {"text": ...}로 최종 변환합니다.
chain = (
    RunnableLambda(mask_privacy)
    | (lambda x: {"text": x})
    | prompt
    | llm
    | parser
)

# 4. Gradio 처리 함수
def process(record):
    # 이제 UI 단에서 마스킹 함수를 조작하지 않습니다.
    # 체인이 실행되면서 자동으로 내부에서 마스킹을 먼저 수행합니다.
    answer = chain.invoke(record)

    # 만약 화면에 마스킹된 결과도 따로 보여주고 싶다면,
    # 독립적으로 함수를 한 번 더 불러서 꽂아주기만 하면 됩니다.
    masked_preview = mask_privacy(record)

    return masked_preview, answer

# 5. UI 구동
demo = gr.Interface(
    fn=process,
    inputs=gr.Textbox(lines=8, label="진료기록 원본 (주민번호·전화·이름 포함)"),
    outputs=[
        gr.Textbox(lines=8, label="🔒 마스킹 결과 (모델에 실제 전달된 내용)"),
        gr.Textbox(lines=8, label="📋 환자 안내문 (최종 결과)"),
    ],
    title="진료기록 안전 변환기",
    description="RunnableLambda를 통해 체인 내부에서 보안 마스킹을 강제 처리합니다.",
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


#### 환자: 김민수 (50세 남성)
#### 주민등록번호: 750312-1234567
#### 연락처: 010-1234-5678
#### 진단: 급성 위염